# **TFM: Detecció d'esdeveniments importants en partits de futbol a partir de les seves narracions**

**Autor:** Martí Mullor Rordíguez

**Institució:** Universitat Oberta de Catalunya  
**Tutor:** Josep Mª Carmona Leyva

**Data:** Juny 2026

---

### **Nom de l'script: 01_Preproces**

En aquest script es dur a terme el preprocés de les dades.

Està compost per:

**0. Importacions**

**1. Configuració inicial**

1.1 Mutar Google Drive

1.2 Carregar la configuració inicial

1.3 Definició de rutes i extracció de parámetres


**2. Descàrrega de les etiquetes, transcripcions i vídeo**

2.1. Descàrrega de les etiquetes (SoccerNer)

2.2. Descàrrega de les Transcripcions de Text (SoccerNet-Echoes)

2.3. Descàrrega del vídeos (SoccerNet)

**3. Funcions d'extracció de característiques d'àudio**

3.1. Extracció d'Àudio amb FFmpeg

3.2. Inicialització del Model Wav2Vec2

3.3. Generació d'Audio Embeddings


**4. Funcions de lectura i alineació**

4.1. Lectura de les etiquetes de Soccer Net

4.2. Alineació temporal de transcripcions i àudio

**5. Processament**

**6. Depuració, divisió del datset i exportació**

6.1. Depuració de segments sense àudio

6.2. Divisió

6.3. Exportació










---


## **0. Importacions**

In [ ]:
import os
import json
import glob
import shutil
import subprocess
import pandas as pd
import numpy as np
import librosa
import torch
import torchaudio

from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from google.colab import drive


## **1. Configuració inicial**

### **1.1. Muntar Google Drive**



In [ ]:
if not os.path.exists('/content/drive'):
    print("Muntant Google Drive")
    drive.mount('/content/drive')

### **1.2. Carregar la configuració inicial**

In [ ]:
CONFIG_PATH = "/content/drive/MyDrive/TFM/TFM-Deteccio-Esdeveniments-Futbol/config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

### **1.3. Definició de rutes i extracció de parámetres**

In [ ]:
paths = config["paths"]
seed = config["global_settings"]["random_seed"]
settings = config["fase_01_preproces"]

LOCAL_RAW_DIR = paths["local_raw"]
LOCAL_TEMP_ECHOES = paths["local_temp_echoes"]
os.makedirs(LOCAL_RAW_DIR, exist_ok=True)

AUDIO_MODEL_NAME = settings["audio_embedding_model"]
TARGET_SR = settings["target_sample_rate"]

train_ratio = settings["train_ratio"]
val_ratio = settings["val_ratio"]

## **2. Descàrrega de les etiquetes, transcripcions i vídeo**

In [ ]:
!pip install SoccerNet -q
from SoccerNet.Downloader import SoccerNetDownloader

# Inicialització del descarregador oficial
downloader = SoccerNetDownloader(LocalDirectory=paths["raw_data"])
downloader.password = settings["soccernet_password"]

In [ ]:
downloader_local = SoccerNetDownloader(LocalDirectory=LOCAL_RAW_DIR)
downloader_local.password = settings["soccernet_password"]

### **2.1. Descàrrega de les etiquetes (SoccerNet)**

In [ ]:
labels_check = glob.glob(os.path.join(LOCAL_RAW_DIR, "**/Labels-v2.json"), recursive=True)
if len(labels_check) == 0:
    print("No s'han trobat etiquetes locals. Descarregant Labels-v2.json...")
    # Descàrrega dels splits principals
    downloader.downloadGames(files=["Labels-v2.json"], split=["train", "valid", "test"])
    print("Etiquetes desades a la carpeta RAW.")


### **2.2. Descàrrega de les Transcripcions de Text (SoccerNet-Echoes)**

In [ ]:
WHISPER_VERSION = settings["whisper_version"]
ECHOES_GIT_URL = settings["echoes_git_url"]

json_files = glob.glob(os.path.join(LOCAL_RAW_DIR, "**/*.json"), recursive=True)
commentary_jsons = [j for j in json_files if "Labels" not in os.path.basename(j)]

if len(commentary_jsons) == 0:
    print(f" Descarregant SoccerNet-Echoes ({WHISPER_VERSION})")

    if os.path.exists(LOCAL_TEMP_ECHOES):
        shutil.rmtree(LOCAL_TEMP_ECHOES)

    # Clonació del repostiroi
    !git clone {ECHOES_GIT_URL} {LOCAL_TEMP_ECHOES} -q

    src_dataset_specific = os.path.join(LOCAL_TEMP_ECHOES, "Dataset", WHISPER_VERSION)

    if os.path.exists(src_dataset_specific):
        !cp -r {src_dataset_specific}/* {LOCAL_RAW_DIR}/
        print("Estructura unificada correctament.")


        shutil.rmtree(LOCAL_TEMP_ECHOES)
    else:
        print(f"ERROR: No s'ha trobat la carpeta {WHISPER_VERSION} al repositori clonat.")
else:
    print(f"S'han detectat {len(commentary_jsons)} fitxers de comentaris de text ja alineats.")


### **2.3. Descàrrega del vídeos (SoccerNet)**


In [ ]:
if settings.get("download_videos", False):
    res = settings["video_resolution"]
    print(f"S'ha activat la descàrrega de vídeos ({res}) al disc LOCAL.")
    try:
        # Descarregador local en lloc del de Drive. S'agafa només valid per reduir el tamany de la mostra.
        downloader_local.downloadGames(files=[f"1_{res}.mkv", f"2_{res}.mkv"], split=["test"])
        print("Descàrrega de fitxers de vídeo finalitzada en local.")
    except Exception as e:
        print(f"Nota de descàrrega de vídeo: {e}.")

## **3. Funcions d'extracció de característiques d'àudio**

### **3.1. Extracció d'Àudio amb FFmpeg**

In [ ]:
def extract_wav_from_video(video_path, output_wav_path):
    """Executa FFmpeg per extreure l'àudio en format WAV si es disposa del vídeo."""
    if os.path.exists(output_wav_path) or not os.path.exists(video_path):
        return
    print(f"Extreient àudio des de: {os.path.basename(video_path)}")
    command = f"ffmpeg -i '{video_path}' -vn -acodec pcm_s16le -ar 16000 -ac 1 '{output_wav_path}' -y"
    subprocess.run(command, shell=True)


### **3.2. Inicialització del Model Wav2Vec2**

In [ ]:
print(f"Carregant el model de característiques acústiques: {AUDIO_MODEL_NAME}...")

audio_processor = Wav2Vec2Processor.from_pretrained(AUDIO_MODEL_NAME)
audio_model = Wav2Vec2Model.from_pretrained(AUDIO_MODEL_NAME).to("cuda")

## **4. Funcions de lectura i alineació**

### **4.1. Lectura de les etiquetes de Soccer Net**

In [ ]:
def parse_soccernet_labels(labels_json_path):
    events = []
    if not os.path.exists(labels_json_path): return events
    with open(labels_json_path, "r", encoding="utf-8") as f: data = json.load(f)
    for annotation in data.get("annotations", []):
        game_time = annotation.get("gameTime", "")
        half = int(game_time.split(" - ")[0]) if " - " in game_time else 1
        position_sec = int(annotation.get("position", 0)) / 1000.0
        events.append({"half": half, "seconds": position_sec, "label": annotation.get("label", "Unknown")})
    return events

### **4.2. Alineació temporal de transcripcions i àudio**

In [ ]:
def align(events, transcripts, half_num, waveform, background_label="background", chunk_size=20, margin=10, stride=10, text_window=15, bg_stride=200):
    """
    Funció per alinear esdeveniments, augmentar dades i generar mostres de classe 'background' equilibrades.
    Optimitzant amb Batching per a GPU i Submostreig de Background.
    """
    augmented_dataset = []
    audio_segments_to_process = []

    half_events = [ev for ev in events if ev["half"] == half_num]
    event_times = [ev["seconds"] for ev in half_events]

    for ev in half_events:
        ev_time = ev["seconds"]
        label = ev["label"]

        text_start = ev_time - text_window
        text_end = ev_time + text_window

        relevant_text = " ".join([
            t["text"] for t in transcripts
            if not (t["end"] < text_start or t["start"] > text_end)
        ])

        window_starts = np.arange(ev_time - margin, ev_time + margin + 1, stride)

        for start_time in window_starts:
            end_time = start_time + chunk_size
            has_audio = False

            if waveform is not None:
                start_sample = int(max(0, start_time) * TARGET_SR)
                end_sample = int(min(len(waveform) / TARGET_SR, end_time) * TARGET_SR)

                if start_sample < len(waveform) and start_sample < end_sample:
                    segment = waveform[start_sample:end_sample]
                    if len(segment) > 0:
                        has_audio = True
                        audio_segments_to_process.append(segment)

            augmented_dataset.append({
                "start": start_time,
                "end": end_time,
                "label": label,
                "text": relevant_text,
                "anchor_time": ev_time,
                "has_audio": has_audio,
                "audio_vector": None, # S'omplirà més tard
                "_segment_index": len(audio_segments_to_process) - 1 if has_audio else -1
            })

    max_time = transcripts[-1]["end"] if transcripts else 0
    background_starts = np.arange(0, max_time, bg_stride)

    for bg_start in background_starts:
        bg_end = bg_start + chunk_size
        is_background = True

        for ev_time in event_times:
            if not (bg_end < (ev_time - margin) or bg_start > (ev_time + margin)):
                is_background = False
                break

        if is_background:
            bg_text = " ".join([
                t["text"] for t in transcripts
                if not (t["end"] < bg_start or t["start"] > bg_end)
            ])

            has_audio = False
            if waveform is not None:
                start_sample = int(max(0, bg_start) * TARGET_SR)
                end_sample = int(min(len(waveform) / TARGET_SR, bg_end) * TARGET_SR)

                if start_sample < len(waveform) and start_sample < end_sample:
                    segment = waveform[start_sample:end_sample]
                    if len(segment) > 0:
                        has_audio = True
                        audio_segments_to_process.append(segment)

            augmented_dataset.append({
                "start": bg_start,
                "end": bg_end,
                "label": background_label,
                "text": bg_text,
                "anchor_time": None,
                "has_audio": has_audio,
                "audio_vector": None, # S'omplirà més tard
                "_segment_index": len(audio_segments_to_process) - 1 if has_audio else -1
            })

    if len(audio_segments_to_process) > 0:
        batch_size = 16
        all_embeddings = []

        for i in range(0, len(audio_segments_to_process), batch_size):
            batch = audio_segments_to_process[i:i+batch_size]

            inputs = audio_processor(batch, sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

            with torch.no_grad():
                outputs = audio_model(**inputs)

            emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.extend(emb)

        for item in augmented_dataset:
            idx = item.pop("_segment_index", -1)
            if idx != -1:
                item["audio_vector"] = all_embeddings[idx]

    return augmented_dataset

## **5. Processament**

In [ ]:
all_segments_list = []

drive_raw_dir = config["paths"]["raw_data"]
labels_files = glob.glob(os.path.join(drive_raw_dir, "**/Labels-v2.json"), recursive=True)

print("\nFiltrant partits... (Buscant vídeos locals)")
test_labels_files = []

for labels_path in labels_files:
    game_drive_dir = os.path.dirname(labels_path)
    rel_path = os.path.relpath(game_drive_dir, drive_raw_dir)
    game_local_dir = os.path.join(LOCAL_RAW_DIR, rel_path)

    # Comprovem si hi ha algun arxiu .mkv en aquest directori local concret
    mkv_files = glob.glob(os.path.join(game_local_dir, "*.mkv"))
    if len(mkv_files) > 0:
        test_labels_files.append(labels_path)

print(f"S'han trobat {len(test_labels_files)} partits amb vídeo local d'un total de {len(labels_files)}.")
print("Començant l'alineació i l'extracció d'àudio...\n")


total_vectors = 0
vectors_exits = 0
vectors_fallits = 0

for labels_path in tqdm(test_labels_files, desc="Processant partits"):
    game_drive_dir = os.path.dirname(labels_path)
    rel_path = os.path.relpath(game_drive_dir, drive_raw_dir)
    game_id = rel_path
    game_local_dir = os.path.join(LOCAL_RAW_DIR, rel_path)

    game_events = parse_soccernet_labels(labels_path)
    if not game_events: continue

    all_jsons = glob.glob(os.path.join(game_local_dir, "**/*.json"), recursive=True)
    asr_files = [j for j in all_jsons if "_asr.json" in os.path.basename(j)]

    for asr_path in asr_files:
        filename = os.path.basename(asr_path)
        half_num = 1 if "1" in filename else (2 if "2" in filename else None)
        if half_num is None: continue

        video_source = os.path.join(game_local_dir, f"{half_num}_{settings['video_resolution']}.mkv")
        audio_target = os.path.join(game_local_dir, f"{half_num}{settings['audio_extension']}")

        # Si el vídeo de la part corresponent no hi és, saltem a la següent
        if not os.path.exists(video_source):
            continue

        if settings.get("extract_audio_ffmpeg", True):
            extract_wav_from_video(video_source, audio_target)

        waveform = None
        if os.path.exists(audio_target):
            try:
                wf_tensor, sr = torchaudio.load(audio_target)
                if sr != TARGET_SR:
                    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=TARGET_SR)
                    wf_tensor = resampler(wf_tensor)
                waveform = wf_tensor.squeeze().numpy()
            except Exception as e:
                print(f"\n[ERROR] Torchaudio ha fallat en carregar l'arxiu {audio_target}: {e}")

        # Iniciem l'alineació
        transcripts = []
        if os.path.exists(asr_path):
            with open(asr_path, "r", encoding="utf-8") as f:
                data = json.load(f)

                if isinstance(data, dict) and "segments" in data:
                    segments_data = data["segments"]

                    if isinstance(segments_data, dict):
                        for seg_list in segments_data.values():
                            if isinstance(seg_list, list) and len(seg_list) >= 3:
                                transcripts.append({
                                    "start": float(seg_list[0]),
                                    "end": float(seg_list[1]),
                                    "text": str(seg_list[2])
                                })

            transcripts.sort(key=lambda x: x["start"])

        # Passem la variable waveform només si està ben carregada
        half_rows = align(game_events, transcripts, half_num, waveform, settings["background_label"])

        for row in half_rows:
            row["game_id"] = game_id
            all_segments_list.append(row)

            if row["has_audio"]:
                total_vectors += 1
                if row["audio_vector"] is not None:
                    vectors_exits += 1
                else:
                    vectors_fallits += 1

                if total_vectors % 500 == 0:
                    print(f"\n-> Progrés: {total_vectors} vectors acústics generats (GPU OK: {vectors_exits} | Errors: {vectors_fallits})")

        # Neteja de l'arxiu temporal
        if os.path.exists(audio_target):
            os.remove(audio_target)
        del waveform

## **6. Depuració, divisió del datset i exportació**

### **6.1. Depuració de segments sense àudio**

In [ ]:
df_master = pd.DataFrame(all_segments_list)

print(f"Mida original del dataset abans de la neteja: {df_master.shape}")

# Es filtra pels fragments que tenen àudio
df_master = df_master[df_master["has_audio"] == True].copy()

# I es filtra també per deixar fora els fragments ocn l'audio ha fallat
df_master = df_master.dropna(subset=['audio_vector']).copy()

print(f"Mida del dataset després de la neteja: {df_master.shape}\n")

In [ ]:
df_master.head()

### **6.2. Divisió**

In [ ]:
np.random.seed(seed)

# S'obtenen i es barrejen els partits

unique_games = df_master['game_id'].unique()
np.random.shuffle(unique_games)

# Càlcul dels índexs de divisió
n_games = len(unique_games)
train_idx = int(n_games * train_ratio)
val_idx = int(n_games * (train_ratio + val_ratio))

train_games = unique_games[:train_idx]
val_games = unique_games[train_idx:val_idx]
test_games = unique_games[val_idx:]

print(f"Total partits: {n_games} | Train: {len(train_games)} | Val: {len(val_games)} | Test: {len(test_games)}")

# Separació del dataframe original
df_train = df_master[df_master['game_id'].isin(train_games)].copy()
df_val = df_master[df_master['game_id'].isin(val_games)].copy()
df_test = df_master[df_master['game_id'].isin(test_games)].copy()



print(f"\nDimensions de Train: {df_train.shape}")
print(f"Dimensions de Val: {df_val.shape}")
print(f"Dimensions de Test: {df_test.shape}")


### **6.3. Exportació**

In [ ]:
# Exportació a Google Drive
df_train.to_pickle(os.path.join(paths["processed_data"], "train_dataset.pkl"))
df_val.to_pickle(os.path.join(paths["processed_data"], "val_dataset.pkl"))
df_test.to_pickle(os.path.join(paths["processed_data"], "test_dataset.pkl"))

print(f"Fitxers guardats correctament a: {paths['processed_data']}")

In [ ]:
import os
import glob

# Utilitzem les mateixes variables que ja tens definides al teu script
drive_raw_dir = config["paths"]["raw_data"]
labels_files = glob.glob(os.path.join(drive_raw_dir, "**/Labels-v2.json"), recursive=True)

print("Auditant els directoris per trobar els partits descartats...\n")
print("-" * 60)

partits_amb_video_inicial = []
partits_problematic = []

for labels_path in labels_files:
    game_drive_dir = os.path.dirname(labels_path)
    rel_path = os.path.relpath(game_drive_dir, drive_raw_dir)
    game_local_dir = os.path.join(LOCAL_RAW_DIR, rel_path)

    # 1. Filtre inicial: té algun arxiu .mkv?
    if len(glob.glob(os.path.join(game_local_dir, "*.mkv"))) > 0:
        partits_amb_video_inicial.append(rel_path)

        problemes = []

        # 2. Comprovar les transcripcions (ASR)
        all_jsons = glob.glob(os.path.join(game_local_dir, "**/*.json"), recursive=True)
        asr_files = [j for j in all_jsons if "_asr.json" in os.path.basename(j)]

        if len(asr_files) == 0:
            problemes.append("Manca de text: No s'ha trobat cap arxiu '_asr.json'")
        else:
            # 3. Comprovar si tenim els vídeos exactes per a les parts que tenen ASR
            for asr_path in asr_files:
                filename = os.path.basename(asr_path)
                half_num = 1 if "1" in filename else (2 if "2" in filename else None)

                if half_num:
                    video_source = os.path.join(game_local_dir, f"{half_num}_{settings['video_resolution']}.mkv")
                    if not os.path.exists(video_source):
                        problemes.append(f"Manca de vídeo: L'arxiu '{os.path.basename(video_source)}' no existeix, tot i tenir text per la part {half_num}")

        # Si hem detectat algun problema que farà saltar el codi principal, ho guardem
        if problemes:
            partits_problematic.append({
                "game_id": rel_path,
                "motius": problemes
            })

# --- Resultats ---
print(f"Total de partits pre-filtrats (amb algun MKV): {len(partits_amb_video_inicial)}")
print(f"Partits complets (amb vídeo de les dues parts i text): {len(partits_amb_video_inicial) - len(partits_problematic)}")
print(f"Partits descartats: {len(partits_problematic)}\n")

print("--- LLISTAT DETALLAT DELS PARTITS DESCARTATS ---")
for p in partits_problematic:
    print(f"\nDirectori: {p['game_id']}")
    for motiu in p['motius']:
        print(f"  -> {motiu}")

In [ ]:
import os
import glob
from SoccerNet.utils import getListGames

# Obtenim les llistes oficials de SoccerNet per a cada partició
train_games = getListGames(["train"])
valid_games = getListGames(["valid"])
test_games = getListGames(["test"])

# Definim la ruta on vas guardar les transcripcions (LOCAL_RAW_DIR)
local_dir = LOCAL_RAW_DIR

def analitza_split(nom_split, llista_partits, base_dir):
    partits_amb_asr = 0

    for game in llista_partits:
        # La llibreria retorna el nom del partit (ex: "england_epl/2014-2015/... ")
        game_local_dir = os.path.join(base_dir, game)

        # Busquem si aquest partit té arxius de transcripció
        asr_files = glob.glob(os.path.join(game_local_dir, "*_asr.json"))
        if len(asr_files) > 0:
            partits_amb_asr += 1

    total = len(llista_partits)
    proporcio = (partits_amb_asr / total) * 100 if total > 0 else 0

    print(f"[{nom_split.upper()}] {partits_amb_asr} partits amb text de {total} ({proporcio:.2f}%)")

print("--- DISPONIBILITAT DE TRANSCRIPCIONS PER CATEGORIA ---")
analitza_split("Train", train_games, local_dir)
analitza_split("Valid", valid_games, local_dir)
analitza_split("Test", test_games, local_dir)